<style>
    /* we can use injections to style the resulting pdf */

    /* wrapping */
    pre, .jp-CodeCell .jp-Editor {
        display: block!important;
    }
    .jp-Cell-inputArea pre {
           page-break-inside: avoid !important;
    }
    .cm-editor.cm-s-jupyter .highlight pre {
        white-space: pre-wrap !important;
    }

    /* margins */
    div#notebook-container,
    div.container,
    div#notebook,
    .jp-Notebook {
        max-width: none !important;
        width: 102.6% !important;
        margin-left: -2.6% !important;
        padding: 0 !important;
    }
    .jp-MarkdownCell {
        margin-left: -6px;
        margin-bottom: 8px;
        margin-top: 12px;
    }
    .jp-OutputArea .jp-RenderedText {
        padding-left: calc(1ch + 16px);
        padding-top: 4px;
        padding-bottom: 8px;
    }
</style>

# Homework #4 (part 1). Exploratory Data Analysis

This document represents the first part of exploratory data analysis - some sanity checks and high-level overviews.

### 1. QUALITY CHECKS

In the previous stage of work (downloading and understanding the dataset), as part of initial preparation, the downloaded data underwent basic cleaning and preparation, which included checking the expected structure of the dataset, filtering the necessary columns, deduplication, ensuring categorical consistency, and initial normalization (full information is provided in data_preparation.ipynb).

Within the scope of this detailed quality analysis, some previously made decisions were revised. This includes:

##### Expansion of the essential columns set:
For comments, the following columns have been added:
1. `controversiality` (integer, 0 or 1), which can be particularly useful for identifying contentious topics, ethical debates, and norm violations
2. `distinguished` (string, e.g., 'moderator'), which is useful for finding official announcements, rule clarifications, and norm enforcement.

For submissions, the following columns have been added:
1. `link_flair_text` (string), a human-labeled category can be used as powerful categorical variables
2. `upvote_ratio` (float), to measure contention at the submission level
3. `domain` (string) to be able to find out the domain for submissions that are links, which can help with mapping the subculture's ecosystem
distinguished
4. `distinguished` (string, e.g., 'moderator') - same as for comments

##### Revision of zero values handling:
The previous approach combined `[removed]`, `[deleted]`, and `''`, which are conceptually different markers, into a single category np.nan. We can assume that these elements convey different signals:

* `''` provides structural information about the type of post (if this is not an error, then no text was ever intended to be here)
* `[removed]` shows community governance signal (the post had text, but a mod deleted it, active moderation is happening)
* `[deleted] `shows user behavior signal (the post had text, but the user deleted it, self-censorship or regret)

Thus, it was decided to remove this stage from the preparation phase and work through it in detail here.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
comments_df = pd.read_parquet('all_comments_cleaned_NEW2.parquet')
submissions_df = pd.read_parquet('all_submissions_cleaned_NEW2.parquet')

In [3]:
comments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3955713 entries, 0 to 3955712
Data columns (total 10 columns):
 #   Column            Dtype         
---  ------            -----         
 0   id                object        
 1   link_id           object        
 2   parent_id         object        
 3   created_utc       datetime64[ns]
 4   author            object        
 5   subreddit         object        
 6   body              object        
 7   score             int64         
 8   controversiality  int64         
 9   distinguished     object        
dtypes: datetime64[ns](1), int64(2), object(7)
memory usage: 301.8+ MB


In [4]:
print(f"Date range: {comments_df['created_utc'].min()} to {comments_df['created_utc'].max()}")

Date range: 2023-05-01 00:00:04 to 2025-04-30 23:59:49


In [5]:
print(comments_df.sample(3))

              id  link_id parent_id         created_utc            author  \
3918360  kevyc5h  18qbn2t   18qbn2t 2023-12-25 18:16:18  Nosequeponer6444   
1494856  kcgzpwz  18c96jt   kcg0m0e 2023-12-08 06:41:35      lonelyrascal   
3743390  jy7ecaw  1642585   jy7dwf1 2023-08-29 08:42:58  pizza_guy_dwight   

           subreddit                                               body  \
3918360  CharacterAI  I guess I might be in danger of getting kidnapped   
1494856       lonely                                                😂😂😂   
3743390  CharacterAI  https://preview.redd.it/6u5qaaa0i0lb1.jpeg?wid...   

         score  controversiality distinguished  
3918360      2                 0          None  
1494856      2                 0          None  
3743390      1                 0          None  


In [6]:
submissions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392439 entries, 0 to 392438
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   id               392439 non-null  object        
 1   created_utc      392439 non-null  datetime64[ns]
 2   author           392439 non-null  object        
 3   subreddit        392439 non-null  object        
 4   title            392439 non-null  object        
 5   selftext         392439 non-null  object        
 6   score            392439 non-null  int64         
 7   upvote_ratio     392439 non-null  float64       
 8   link_flair_text  392439 non-null  object        
 9   domain           392439 non-null  object        
 10  distinguished    392439 non-null  object        
 11  num_comments     392439 non-null  int64         
 12  over_18          392439 non-null  object        
 13  url              392439 non-null  object        
 14  permalink        392

In [7]:
print(submissions_df.sample(3))

             id         created_utc            author    subreddit  \
367348  1jw6h85 2025-04-10 19:29:14      annonymus_me  CharacterAI   
270540  15bsn9j 2023-07-28 09:35:33  theallmightyrick  CharacterAI   
332790  1bw13gf 2024-04-04 22:30:10       f88x76w8n58  CharacterAI   

                                                    title  \
367348                                       What‘s this?   
270540  So I Somehow Managed To Break through the Dumb...   
332790                             Who would expect that?   

                                                 selftext  score  \
367348  Does anyone else had this before? I‘m very con...    602   
270540                                                         1   
332790                                                         5   

        upvote_ratio      link_flair_text     domain distinguished  \
367348          0.96  Discussion/Question  i.redd.it          None   
270540          1.00          SCREENSHOTS  i.redd.it         

#### 1.1. Null handling

In [8]:
print("Comments Nulls:")
print(comments_df.isnull().sum())

Comments Nulls:
id                  0
link_id             0
parent_id           0
created_utc         0
author              0
subreddit           0
body                0
score               0
controversiality    0
distinguished       0
dtype: int64


In [9]:
print("Submissions Nulls:")
print(submissions_df.isnull().sum())

Submissions Nulls:
id                 0
created_utc        0
author             0
subreddit          0
title              0
selftext           0
score              0
upvote_ratio       0
link_flair_text    0
domain             0
distinguished      0
num_comments       0
over_18            0
url                0
permalink          0
dtype: int64


A basic check shows that there are no null values. This is to be expected, given the above-described rejection of null processing during the basic preparation stage. It also allows us to verify that other columns do not contain unexpected null values.

But, as stated before, differentiating between `[removed]` (moderator) and `[deleted]` (user) is crucial. For a more detailed review, we'll perform an analysis of common placeholders. Hypothesis here: `[removed]` and `[deleted]` are the primary placeholders, but other automated or templated text might exist.

In [10]:
# we filter for comments with some substance to avoid single-word answers ("yes", "no")
# but not so long that they are unlikely to be templates

print("Top 20 most frequent, non-trivial comment bodies:")
comment_bodies = comments_df['body']
frequent_comments = comment_bodies[comment_bodies.str.len().between(5, 200)].value_counts().head(20)
print(frequent_comments)

Top 20 most frequent, non-trivial comment bodies:
body
[removed]                   171987
[deleted]                    46355
Thank you!                    2202
Thanks!                       1321
Thank you                     1113
Thanks                        1030
Good bot                       692
What?                          556
Based                          483
Same here                      419
Exactly                        417
Happy birthday                 396
Me too                         384
Thank you.                     370
Agreed                         344
Same.                          343
Dm me                          328
I also left a review! :D       302
Interesting                    296
Thank you so much!             292
Name: count, dtype: int64


In [11]:
print("Top 20 most frequent, non-trivial submission titles:")
submission_texts = submissions_df['title']
frequent_submissions = submission_texts[submission_texts.str.len().between(5, 200)].value_counts().head(20)
print(frequent_submissions)

Top 20 most frequent, non-trivial submission titles:
title
[image processing failed]               332
Site down                               155
What?                                   141
Question                                137
Lonely                                  136
Title                                   103
Anyone up for chat? Indian male here    100
Excuse me?                               90
Hello                                    83
Here we go again                         82
Site down?                               79
NOT AGAIN                                78
I need help                              72
What.                                    69
[ Removed by Reddit ]                    69
Help?                                    65
Anyone else?                             64
NOOOO                                    63
Anyone wanna chat and be friends?        62
ITS BACK                                 56
Name: count, dtype: int64


In [12]:
print("Top 20 most frequent, non-trivial submission selftexts:")
submission_texts = submissions_df['selftext']
frequent_submissions = submission_texts[submission_texts.str.len().between(5, 200)].value_counts().head(20)
print(frequent_submissions)

Top 20 most frequent, non-trivial submission selftexts:
selftext
[removed]                                                                                                                                                             57102
[deleted]                                                                                                                                                             14067
Title                                                                                                                                                                    69
Indian male here                                                                                                                                                         62
title                                                                                                                                                                    42
Here on Writing Weekends, share what you are going to write and are current

In [13]:
# an example of such short comments that can be marked in the future
print("Low-content comments")
print(f"Comments with 1-3 characters: {len(comment_bodies[comments_df['body'].str.len().between(1, 3)])}\n")
print(f"Example of short comments:\n{comments_df[comments_df['body'].str.len().between(1, 3)]['body'].value_counts().head(20)}")

Low-content comments
Comments with 1-3 characters: 46261

Example of short comments:
body
Yes    2555
No     1754
Lol    1422
lol     826
💀       821
😂       724
No.     722
👍       719
Hi      676
yes     649
Ok      585
Yep     558
no      522
?       498
Fr      484
🫂       437
🤣       416
LOL     407
Wow     405
😭       383
Name: count, dtype: int64


Besides the expected results, we see a recurring pattern of posts, probably from r/lonely (Indian man here...). Let's check it out right away.

In [14]:
# use .str.contains() to catch all variations of this pattern
lonely_pattern_df = submissions_df[submissions_df['selftext'].str.contains("Indian male here", na=False)]
print(f"Found {len(lonely_pattern_df)} posts containing this pattern. Subreddit distribution for these posts:")
print(lonely_pattern_df['subreddit'].value_counts())

Found 112 posts containing this pattern. Subreddit distribution for these posts:
subreddit
lonely    112
Name: count, dtype: int64


This is probably a form of subcultural norm within that specific community. Although it is not relevant to our analysis, it may be useful in later stages, such as keyword filtering to eliminate noise.

More critically, we see the format `“Removed by Reddit on account of violating...”`. Unlike other types of deleted content markers, this one is of particular interest - it is an admin action, that is performed after a violation of site-wide content policy. This is reserved for serious offenses like illegal content, targeted harassment, copyright infringement, etc. Let's investigate this in more detail:

In [15]:
admin_removed_text = "[ Removed by Reddit on account of violating the [content policy](/help/contentpolicy). ]"
admin_removed_df = submissions_df[submissions_df['selftext'] == admin_removed_text]

print(f"Found {len(admin_removed_df)} posts that were removed by reddit admins\n")

print('Subreddit distribution for these posts:')
print(admin_removed_df['subreddit'].value_counts())

print('\nDataset example:')
print(admin_removed_df[['subreddit', 'title', 'score', 'link_flair_text', 'num_comments']])

Found 10 posts that were removed by reddit admins

Subreddit distribution for these posts:
subreddit
lonely        6
ChatGPT       3
FanFiction    1
Name: count, dtype: int64

Dataset example:
         subreddit                                              title  score  \
17375       lonely                              [ Removed by Reddit ]      0   
37232   FanFiction                              [ Removed by Reddit ]      0   
48836       lonely                    My best friend doesn't love me.      2   
50298       lonely  [16/F] here not really sure what I'm doing her...      0   
61227      ChatGPT  The best ChatGPT tools without API Key [ChatGP...      3   
75810       lonely                                     please kill me      1   
84990      ChatGPT                            AI Undetectable Writing      1   
123956      lonely  Thinking about hiring a male escort (I’m a woman)      1   
124242      lonely  the next time someone cancels or flakes on me ...      1   
140789 

We see another similar format - `[Removed by Reddit]` for tittle. Let's analyze it as well.

In [16]:
admin_removed_text = "[ Removed by Reddit ]"
admin_removed_df2 = submissions_df[submissions_df['title'] == admin_removed_text]

print(f"Found {len(admin_removed_df2)} posts that were removed by reddit admins.\n")

print('Subreddit distribution for these posts:')
print(admin_removed_df2['subreddit'].value_counts())

print('\nDataset example:')
print(admin_removed_df2[['subreddit', 'selftext', 'score', 'link_flair_text', 'num_comments']] \
    .sort_values(by='score', ascending=False) \
    .head(10))

Found 69 posts that were removed by reddit admins.

Subreddit distribution for these posts:
subreddit
lonely         22
CharacterAI    21
ChatGPT        18
singularity     5
FanFiction      2
LocalLLaMA      1
Name: count, dtype: int64

Dataset example:
          subreddit   selftext  score      link_flair_text  num_comments
209883   LocalLLaMA  [removed]    166                Other            51
200942  CharacterAI  [removed]     28          🔥SITE DOWN🔥             3
41502        lonely  [removed]      4           Discussion            36
214158  CharacterAI  [removed]      3                 None             0
10131        lonely  [removed]      2                 None             0
367807  CharacterAI  [removed]      1  Discussion/Question             0
225811  CharacterAI  [removed]      1                 None             0
225833  CharacterAI  [removed]      1                 None             0
5493        ChatGPT  [removed]      1           Gone Wild              2
6019    singular

The analysis was performed on the above datasets in full; for the report, they are presented in a shortened format.

A significant portion of admin-removed posts originates from r/lonely. The titles and the flairs point to users in extreme emotional distress -
reddit admins are likely intervening here not just for policy violations, but also as part of their "Reddit Cares" protocol. In a sense, this may provide a context for why people seek AI companionship, as a response to profound isolation and despair. This validates the inclusion of r/lonely as a crucial source of user motivation.

The r/ChatGPT and r/singularity show a different pattern - the flairs (Jailbreak, Gone Wild, Educational Purpose Only) and titles (AI Undetectable Writing) suggest users are pushing (intentionally or unintentionally) the boundaries of what is permissible with AI. This is important information in the context of our analysis of “unintended uses,” which will be explored in more detail in the following sections of the analysis.

The r/CharacterAI and r/LocalLLaMA do not fall under such broad categories. In the case of r/CharacterAI, as a platform with its own safety filters, admin removals could be related to users sharing methods to exploit the platform in ways that violate Reddit's TOS. Considering flair (🔥SITE DOWN🔥), perhaps a user posted something malicious or illegal during a chaotic period.

r/LocalLLaMA presents the most sudden and most popular result among similar posts (166 upvotes and 51 comments). Given the lack of other information and the uninformative link_flair_text, we can weakly assume that this could be a discussion about a dataset or model that contained illegal material or was trained on copyrighted data in a way that prompted a DMCA takedown.

In addition to providing information for further analysis, this stage allowed us to identify the full hierarchy of removal states. Based on this, we can derive a new ‘text_status’ categorical column.

In [17]:
def add_text_status_column(df, text_col='body'):
    # choices for categorization
    conditions = [
        df[text_col] == '[removed]',
        df[text_col] == '[deleted]',
        df[text_col].str.contains('Removed by Reddit', na=False), # catches both variants
        df[text_col].isin(['', None]),
        df[text_col].notna() # any other non-null value
    ]

    choices = [
        'removed_by_moderator',
        'deleted_by_user',
        'removed_by_admin',
        'empty', # post was intentionally blank (e.g., a link post with no selftext)
        'available' # the text is present and valid
    ]

    df['text_status'] = np.select(conditions, choices, default='unknown')

    return df

In [18]:
comments_df = add_text_status_column(comments_df, text_col='body')
submissions_df = add_text_status_column(submissions_df, text_col='selftext')

In [19]:
# a summary of how text content is distributed across these categories
print("Comment text status distribution:")
print(comments_df['text_status'].value_counts(normalize=True))

Comment text status distribution:
text_status
available               0.944782
removed_by_moderator    0.043478
deleted_by_user         0.011718
removed_by_admin        0.000020
empty                   0.000002
Name: proportion, dtype: float64


In [20]:
print("Submission text status distribution:")
print(submissions_df['text_status'].value_counts(normalize=True))

Submission text status distribution:
text_status
available               0.474255
empty                   0.344369
removed_by_moderator    0.145505
deleted_by_user         0.035845
removed_by_admin        0.000025
Name: proportion, dtype: float64



Distribution results for comment text are expected. In later stages, it will be useful to look at the difference in these statistics in terms of subreddits. In the case of submissions, we can observe an unexpectedly ‘empty’ large category.
It is likely that empty selftext corresponds to link/media posts, which can be confirmed using the domain column.

We can filter the submissions dataframe to only include posts where text_status == 'empty' and then examine the top domains for each subreddit.

In [21]:
empty_submissions = submissions_df[submissions_df['text_status'] == 'empty'].copy()
subreddits = empty_submissions['subreddit'].unique()

print(f"Total number of empty submissions: {len(empty_submissions)}")

Total number of empty submissions: 135144


In [22]:
for sub in sorted(subreddits):
    print(f"\nTop {10} domains for r/{sub}")

    sub_df = empty_submissions[empty_submissions['subreddit'] == sub]
    domain_counts = sub_df['domain'].value_counts()
    print(domain_counts.head(10))


Top 10 domains for r/ArtificialInteligence
domain
i.redd.it              118
reddit.com             114
youtube.com             68
self.ChatGPT            58
self.enoumen            37
youtu.be                35
techstormai.com         35
self.singularity        34
self.datascience        30
self.Jhonjournalist     30
Name: count, dtype: int64

Top 10 domains for r/CharacterAI
domain
i.redd.it            61655
reddit.com            9521
self.CharacterAI      7804
c.ai                   435
v.redd.it              205
beta.character.ai      204
youtube.com            143
youtu.be               131
i.imgur.com             77
tiktok.com              20
Name: count, dtype: int64

Top 10 domains for r/ChatGPT
domain
i.redd.it              17614
reddit.com              6658
self.ChatGPT            2050
v.redd.it               1491
youtu.be                 888
youtube.com              827
                         271
chat.openai.com          268
elblogdefamosas.com      231
i.imgur.com       

We can identify certain patterns in our tiers:

Tier 1 communities like r/LocalLLaMA act as technical knowledge hubs linking to GitHub, Hugging Face, and arXiv, while r/SillyTavernAI and r/JanitorAI_Official focus on practical use with little external linking. r/CharacterAI users heavily screenshot their platform interactions.Tier 2 and 3 shift from content curation (r/ChatGPT, r/singularity) and aggregation (r/ArtificialIntelligence) to text-based sharing (r/fanfiction). r/lonely does not have similar posts (a feature of the subreddit)

The information is within expectations. Important findings for the data quality check stage here are empty domain entries (“”) and spam domains (elblogdefamosas.com, hotelwarehouse.com).

In [23]:
# investigating the spam domains
spam_domains = ['hotelwarehouse.com', 'elblogdefamosas.com']
spam_posts = empty_submissions[empty_submissions['domain'].isin(spam_domains)]

In [24]:
print(f"Found {len(spam_posts)} posts linking to suspected spam domains. Authors posting these links:")
print(spam_posts['author'].value_counts())

print("\nSubreddits where these links are posted:")
print(spam_posts['subreddit'].value_counts())

Found 473 posts linking to suspected spam domains. Authors posting these links:
author
SuddenMath7596    242
trandanhreddit    231
Name: count, dtype: int64

Subreddits where these links are posted:
subreddit
singularity    242
ChatGPT        231
Name: count, dtype: int64


In [25]:
top_spammer = spam_posts['author'].value_counts().index[0]
print(f"\nExample titles from top spammer '{top_spammer}':")
print(spam_posts[spam_posts['author'] == top_spammer]['title'].head())


Example titles from top spammer 'SuddenMath7596':
42819    About This Item 9oz PP individually wrapped Pl...
42830    About This Item : The high-quality Ironing boa...
43857    Every hotel should have luggage carts to enhan...
46120    About this item Please check your dimensions B...
46922    Energy Saving Efficiency – The Eden Wireless E...
Name: title, dtype: object


In [26]:
top_spammer = spam_posts['author'].value_counts().index[1]
print(f"\nExample titles from top spammer '{top_spammer}':")
print(spam_posts[spam_posts['author'] == top_spammer]['title'].head())


Example titles from top spammer 'trandanhreddit':
32456    10 Artificial Intelligence Tools to Make Your ...
32846    Comic: Artificial Intelligence - Powell Creek ...
49995    Design school dean criticizes Dezeen's artific...
50459    When AI Robots Impersonate Humans Welcome to E...
50916    Dev News: Svelte 4, Deno Updates, AWS AI Fundi...
Name: title, dtype: object


We see examples of evident spam, which is confirmed by link analysis. This is an important insight for further filtering.

For the second step - we assumed that posts with empty text_status are link/media posts, which is now partially confirmed by the statistics above. We will conduct additional analysis to confirm that this is indeed the case in all instances. Formally, an empty selftext can occur when it's either a link-post (which by definition has no selftext), or a self-post where the user wrote a title but left the body blank. For now, both types exist within text_status == 'empty' - we can prove this and separate them.

In [27]:
submissions_df['submission_type'] = submissions_df['domain'].apply(
        lambda x: 'Self-Post' if str(x).startswith('self.') else 'Link/Media Post'
)

In [28]:
verification_crosstab = pd.crosstab(
        submissions_df['text_status'],
        submissions_df['submission_type']
)

In [29]:
print(verification_crosstab)

submission_type       Link/Media Post  Self-Post
text_status                                     
available                       40052     146064
deleted_by_user                 13942        125
empty                          121674      13470
removed_by_admin                    1          9
removed_by_moderator            15134      41968


We can see that for 'empty':
* 121,674 are true Link/Media Posts (as expected).
* 13,470 are Title-Only Self-Posts (hypothesis confirmed)

However, it also presents a new question with the ("available | Link/Media Post | 40052"), which is logically inconsistent with our initial assumption (if a post is a "Link/Media Post," its selftext should be empty). Hypothesis: this is likely due to a newer Reddit feature where users can submit a link and add a body of text, similar to a self-post. We should test this later

We also need to investigate the `""` domain. Let's isolate these posts and look directly at their url and title

In [30]:
anomalous_df = submissions_df[submissions_df['domain'].isin([''])].copy()

In [31]:
print(f"Found {len(anomalous_df)} posts with anomalous domains.\n")

print("Distribution by subreddit:")
print(anomalous_df['subreddit'].value_counts())

Found 18768 posts with anomalous domains.

Distribution by subreddit:
subreddit
CharacterAI              6008
ChatGPT                  5809
lonely                   4202
singularity               926
FanFiction                676
ArtificialInteligence     502
JanitorAI_Official        330
LocalLLaMA                223
SillyTavernAI              81
MyBoyfriendIsAI            11
Name: count, dtype: int64


In [32]:
print(anomalous_df[['subreddit', 'author', 'selftext', 'title', 'domain', 'url', 'permalink']].sample(3))

          subreddit     author   selftext  \
10925       ChatGPT  [deleted]  [removed]   
51569    FanFiction  [deleted]  [deleted]   
212561  CharacterAI  [deleted]  [deleted]   

                                                    title domain url  \
10925              Hahaha it's hallucinating reddit posts              
51569                           Do I turn my oneshot NSFW              
212561  I made a surprisingly wholesome nudist (sorry ...              

                                                permalink  
10925   /r/ChatGPT/comments/13e4zdh/hahaha_its_halluci...  
51569   /r/FanFiction/comments/14iiywe/do_i_turn_my_on...  
212561  /r/CharacterAI/comments/13tbmqt/i_made_a_surpr...  


We see a large number of markers [deleted] and [removed], as well as, partially, [View Poll]. Let's track this in more detail.

In [33]:
print(anomalous_df["author"].value_counts().head(5))

author
[deleted]             18439
Tall_Ad4729             100
Inevitable-Rub8969       17
OtiCinnatus               8
LeveredRecap              7
Name: count, dtype: int64


In [34]:
counts = anomalous_df['author'].value_counts(dropna=False)
summary = counts.reindex(['[deleted]'], fill_value=0)
summary['Other'] = counts[~counts.index.isin(['[deleted]'])].sum()

result_df = (
    summary.to_frame('Count')
    .assign(Percent=lambda x: (x['Count'] / x['Count'].sum() * 100).round(2))
    .sort_values('Count', ascending=False)
)

print(result_df)

           Count  Percent
author                   
[deleted]  18439    98.25
Other        329     1.75


In [35]:
print(anomalous_df["selftext"].value_counts().head(5))

selftext
[deleted]                                                        13080
[removed]                                                         5281
                                                                   327
[removed]\n\n[View Poll](https://www.reddit.com/poll/134b583)        1
[removed]\n\n[View Poll](https://www.reddit.com/poll/134m55f)        1
Name: count, dtype: int64


In [36]:
counts = anomalous_df['selftext'].value_counts(dropna=False)
summary = counts.reindex(['[deleted]', '[removed]'], fill_value=0)
summary['Other'] = counts[~counts.index.isin(['[deleted]', '[removed]'])].sum()

result_df = (
    summary.to_frame('Count')
    .assign(Percent=lambda x: (x['Count'] / x['Count'].sum() * 100).round(2))
    .sort_values('Count', ascending=False)
)

print(result_df)

           Count  Percent
selftext                 
[deleted]  13080    69.69
[removed]   5281    28.14
Other        407     2.17


In [37]:
# to specifically check all the rows where 'author' column is not
# '[deleted]' AND 'selftext' is not '[deleted]' OR '[removed]'

filtered_df = anomalous_df[
    (anomalous_df['author'] != '[deleted]') &
    (~anomalous_df['selftext'].isin(['[deleted]', '[removed]']))
]
print(filtered_df[['subreddit', 'title', 'domain', 'url', 'permalink']].sample(3))

       subreddit                                              title domain  \
162217   ChatGPT  ChatGPT Prompt of the Day: 🔥 GHOST LEDGER: QUA...          
181190   ChatGPT  Does anybody know if there's been any talk abo...          
174690   ChatGPT  AI helped me ace my final thesis with just one...          

                                                      url  \
162217  /r/ChatGPTPromptGenius/comments/1jrpdv1/chatgp...   
181190  /r/OpenAI/comments/1k3iyr4/does_anybody_know_i...   
174690  /r/AGI_News/comments/1jztnjz/ai_helped_me_ace_...   

                                                permalink  
162217  /r/ChatGPT/comments/1jrpdzw/chatgpt_prompt_of_...  
181190  /r/ChatGPT/comments/1k3wb2a/does_anybody_know_...  
174690  /r/ChatGPT/comments/1jztppc/ai_helped_me_ace_m...  


Considering that the permalink is the location of the submission in the current subreddit (e.g., r/ChatGPT), and the url is a relative path to the original submission in a different subreddit (e.g., /r/ChatGPTPromptGenius/..) we can draw an important conclusion for further analysis - crossposts have this structure.

We can probably also assume that when a post is deleted by a user or removed by a moderator, key metadata fields like url and domain are often wiped clean, becoming empty strings.

This creates two distinct pathways that lead to a post having an empty domain:
* For crossposts, the url is a relative path (e.g., /r/Subreddit/...), and the domain is parsed as "". The author and selftext are intact.
* When a user creates a standard link post (e.g., to i.redd.it or youtube.com) and later, the post is deleted or removed. In the Pushshift data, the author becomes [deleted], the selftext becomes [deleted] or [removed], and the original url and domain are obliterated and replaced with "".

This should also be analyzed in more detail in the later part of the task, including [View Poll] labels  for poll posts. Also, considering all of the above information, our data model needs an upgrade to account for all these states, which will also be done in the next part.